# Evaluación de objetos y relaciones espaciales

SUN RGB-D se utiliza para detección y recuperación informada por objetos. Visual Genome filtrado a interiores se utiliza únicamente para contrastar reglas 2D con relaciones anotadas.


In [ ]:
from pathlib import Path
import sys

repo = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "semantic_navigation_ws" / "src").is_dir())
sys.path.insert(0, str(repo / "experiments" / "shared"))

import numpy as np
import pandas as pd

from notebook_bootstrap import bootstrap_offline, resolve_repo_path
from semantic_evaluation.core.dataset_adapters import load_dataset

ctx = bootstrap_offline()
config = ctx["config"]
specs = {spec.dataset_id: spec for spec in ctx["dataset_specs"]}
sunrgbd = load_dataset(specs["sunrgbd"], ctx["repo_root"])
visual_genome = load_dataset(specs["visual_genome"], ctx["repo_root"])
display(pd.DataFrame([
    {"dataset_id": "sunrgbd", "available": not sunrgbd.skipped,
     "samples": len(sunrgbd.nodes), "reason": sunrgbd.skip_reason},
    {"dataset_id": "visual_genome", "available": not visual_genome.skipped,
     "samples": len(visual_genome.relation_ground_truth), "reason": visual_genome.skip_reason},
]))


## Detección de objetos en SUN RGB-D


In [ ]:
from semantic_evaluation.core import EmbeddingCache
from semantic_evaluation.core.evaluation_statistics import (
    detection_average_precision, match_detections, precision_recall_f1)
from semantic_evaluation.core.offline_encoding import detect_objects
from semantic_vision_core import SemanticVisionPipeline

detection_rows = []
ap_rows = []
mean_average_precision = np.nan
pipeline = None
checkpoint_value = config["models"]["yolo"]["checkpoint"]
checkpoint = None
if "${" not in checkpoint_value:
    checkpoint = resolve_repo_path(ctx["repo_root"], checkpoint_value)
if not sunrgbd.skipped and checkpoint is not None and checkpoint.is_file():
    siglip = config["models"]["siglip"]
    cache = EmbeddingCache(str(resolve_repo_path(
        ctx["repo_root"], config["paths"]["cache_root"]) / "sunrgbd_yolo"))
    try:
        pipeline = SemanticVisionPipeline(
            retrieval_mode="siglip_yolo", siglip_model_id=siglip["model_id"],
            yolo_model_path=str(checkpoint),
            yolo_confidence_threshold=config["models"]["yolo"]["confidence_threshold"],
            device=ctx["device"], processor_fast=siglip["processor_fast"],
            local_files_only=siglip.get("local_files_only", False))
    except (ImportError, OSError, RuntimeError) as error:
        print("Modelos no disponibles:", error)
    if pipeline is not None:
        detect_objects(sunrgbd.nodes, pipeline, cache, str(checkpoint),
                       config["models"]["yolo"]["confidence_threshold"],
                       with_crop_embeddings=True, siglip_model_id=siglip["model_id"])
        predictions = {observation.observation_id: observation.objects
                       for node in sunrgbd.nodes for observation in node.observations}
        mapping = sunrgbd.metadata["class_mapping"]
        for observation_id, truth in sunrgbd.object_ground_truth.items():
            tp, fp, fn, unmapped = match_detections(
                predictions.get(observation_id, []), truth, mapping)
            detection_rows.append({"observation_id": observation_id,
                                   "tp": tp, "fp": fp, "fn": fn,
                                   "unmapped_classes": unmapped,
                                   **precision_recall_f1(tp, fp, fn)})
        ap_rows, mean_average_precision = detection_average_precision(
            predictions, sunrgbd.object_ground_truth, mapping)
elif not sunrgbd.skipped:
    print("YOLO omitido: define YOLO_CHECKPOINT con un fichero local.")
display(pd.DataFrame(detection_rows))
display(pd.DataFrame(ap_rows))
print("mAP@0.5:", mean_average_precision)


## Relaciones 2D en Visual Genome


In [ ]:
from semantic_navigation_core.relations import infer_relations
from semantic_evaluation.core.evaluation_statistics import relation_metrics

predicted_relations = []
annotated_relations = []
relation_examples = []
if not visual_genome.skipped:
    for image_id, truth in visual_genome.relation_ground_truth.items():
        inferred = infer_relations(visual_genome.object_ground_truth.get(image_id, []))
        predicted_relations.extend(inferred)
        annotated_relations.extend(truth)
        relation_examples.append({"image_id": image_id,
                                  "predicted": len(inferred), "annotated": len(truth),
                                  "mean_inferred_confidence": np.mean(
                                      [relation.confidence for relation in inferred]) if inferred else np.nan})
relation_summary = pd.DataFrame(relation_metrics(predicted_relations, annotated_relations))
display(relation_summary)
display(pd.DataFrame(relation_examples).head(20))


## Recuperación informada por objetos


In [ ]:
query_path = resolve_repo_path(ctx["repo_root"], specs["sunrgbd"].queries_file)
from semantic_evaluation.core.offline_dataset import load_queries
sun_queries = load_queries(str(query_path))
if pipeline is None or not sun_queries:
    print("Comparación omitida: requiere SUN RGB-D, modelos locales y consultas con valid_node_ids.")
else:
    print("Las funciones de recuperación existentes se ejecutan en 03_hybrid_evaluation sobre el conjunto común anotado.")


## Exportación e interpretación


In [ ]:
results_root = resolve_repo_path(ctx["repo_root"], config["paths"]["results_root"]) / "objects_and_relations"
results_root.mkdir(parents=True, exist_ok=True)
if relation_summary.empty and not detection_rows:
    print("No se exportan CSV: faltan los datasets o checkpoints indicados arriba.")
else:
    if detection_rows:
        pd.DataFrame(detection_rows).to_csv(results_root / "object_detection.csv", index=False)
        pd.DataFrame(ap_rows).to_csv(results_root / "object_average_precision.csv", index=False)
    if not relation_summary.empty:
        relation_summary.to_csv(results_root / "relation_metrics.csv", index=False)
    if detection_rows:
        totals = pd.DataFrame(detection_rows)[["tp", "fp", "fn"]].sum()
        print("Detecciones acumuladas:", totals.to_dict(), "mAP@0.5:", mean_average_precision)
    if not relation_summary.empty:
        weakest = relation_summary.sort_values("f1").iloc[0]
        print(f"La menor F1 de relaciones fue {weakest['f1']:.3f} para {weakest['predicate']}.")
